# EGFR 변이체 — 약물 내성 도킹 분석

EGFR WT 및 변이체(L858R, T790M, T790M/L858R)에 1~3세대 TKI를 도킹하여
**내성 메커니즘**을 시뮬레이션합니다.

## 워크플로우
1. EGFR 구조 & 약물 패널 정의
2. 구조 준비 & 정렬
3. Docking Box 설정 + 3D 확인
4. Cross-docking (약물 × 변이체)
5. 내성 패턴 분석 (히트맵, 상호작용)
6. 내보내기


## 0. 환경 설정


In [ ]:
#@title 환경 설치 {display-mode: "form"}
import subprocess, sys, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

pip_install('rdkit', 'meeko', 'openbabel-wheel')
pip_install('pdbfixer', 'openmm')
pip_install('py3Dmol', 'prolif', 'MDAnalysis')
pip_install('seaborn', 'pandas', 'matplotlib', 'requests')
try: pip_install('pymol-open-source')
except: pass

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    import stat, urllib.request
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('Done.')


In [ ]:
#@title 라이브러리 로드 {display-mode: "form"}
import warnings; warnings.filterwarnings('ignore')
import os, subprocess, urllib.request, time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, SanitizeFlags
from openbabel import pybel
from meeko import MoleculePreparation, PDBQTWriterLegacy
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda
from MDAnalysis.analysis import align as mda_align
import prolif as plf

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
WORK_DIR = '/content/egfr_mutants' if os.path.exists('/content') else os.path.join(os.path.expanduser('~'), 'egfr_mutants')
os.makedirs(WORK_DIR, exist_ok=True)
print('All libraries loaded.')


## 1. EGFR 구조 & 약물 패널 정의


In [ ]:
#@title 1-1. EGFR 변이체 구조 {display-mode: "form"}

EGFR_STRUCTURES = [
    {'label': 'WT_Erlotinib',    'pdb': '1M17', 'chain': 'A', 'mutation': 'Wild-type',    'generation': '1st gen'},
    {'label': 'L858R_Gefitinib', 'pdb': '2ITZ', 'chain': 'A', 'mutation': 'L858R',        'generation': '1st gen'},
    {'label': 'T790M_Neratinib', 'pdb': '2JIT', 'chain': 'A', 'mutation': 'T790M',        'generation': '2nd gen'},
    {'label': 'DM_WZ4002',       'pdb': '3IKA', 'chain': 'A', 'mutation': 'T790M/L858R',  'generation': '3rd gen-like'},
]

egfr_df = pd.DataFrame(EGFR_STRUCTURES)
egfr_df.index += 1
egfr_df


In [ ]:
#@title 1-2. 약물 패널 정의 {display-mode: "form"}

DRUGS = {
    'Erlotinib':   {'smiles': 'C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1',                                    'gen': '1st', 'mechanism': 'Reversible'},
    'Gefitinib':   {'smiles': 'COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1',                                'gen': '1st', 'mechanism': 'Reversible'},
    'Afatinib':    {'smiles': 'CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OC1CCOC1',                     'gen': '2nd', 'mechanism': 'Irreversible'},
    'Osimertinib': {'smiles': 'C=CC(=O)Nc1cc(OC)c(Nc2nccc(-c3cn(C)c4ccccc34)n2)cc1N(C)CCN(C)C',                'gen': '3rd', 'mechanism': 'Irreversible'},
}

drug_mols = []
drug_labels = []
for name, info in DRUGS.items():
    mol = Chem.MolFromSmiles(info['smiles'])
    if mol:
        drug_mols.append(mol)
        drug_labels.append(f"{name} ({info['gen']})")

Draw.MolsToGridImage(drug_mols, legends=drug_labels, molsPerRow=4, subImgSize=(300, 220))


## 2. 구조 준비 & 정렬


In [ ]:
#@title 2-1. 전체 구조 준비 {display-mode: "form"}
structures = {}

for i, cx in enumerate(EGFR_STRUCTURES):
    pdb_id = cx['pdb'].lower()
    label = cx['label']
    print(f'[{i+1}/{len(EGFR_STRUCTURES)}] {label} ({pdb_id.upper()})...', end=' ', flush=True)
    
    sdir = os.path.join(WORK_DIR, label)
    os.makedirs(sdir, exist_ok=True)
    
    try:
        pdb_path = os.path.join(sdir, f'{pdb_id}.pdb')
        if not os.path.exists(pdb_path):
            urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', pdb_path)
        
        u = mda.Universe(pdb_path)
        prot_sel = u.select_atoms(f'protein and chainID {cx["chain"]}')
        clean_pdb = os.path.join(sdir, f'{pdb_id}_clean.pdb')
        prot_sel.write(clean_pdb)
        
        prot_H = os.path.join(sdir, f'{pdb_id}_clean_H.pdb')
        fixer = PDBFixer(filename=clean_pdb)
        fixer.findMissingResidues(); fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues(); fixer.removeHeterogens(True)
        fixer.findMissingAtoms(); fixer.addMissingAtoms(); fixer.addMissingHydrogens(7.4)
        with open(prot_H, 'w') as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)
        
        rec_qt = os.path.join(sdir, f'{pdb_id}_rec.pdbqt')
        mol_ob = list(pybel.readfile(format='pdb', filename=prot_H))[0]
        out = pybel.Outputfile(filename=rec_qt+'.tmp', format='pdbqt', overwrite=True)
        out.write(mol_ob); out.close()
        with open(rec_qt+'.tmp') as f: raw = f.readlines()
        with open(rec_qt, 'w') as f:
            for l in raw:
                if not l.startswith(('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')): f.write(l)
        os.remove(rec_qt+'.tmp')
        
        # Co-crystal ligand coords for box
        lig_sel = u.select_atoms(f'not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4 and chainID {cx["chain"]}')
        if len(lig_sel) < 3:
            lig_sel = u.select_atoms('not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4')
        
        structures[label] = {
            'pdb_id': pdb_id, 'prot_H': prot_H, 'rec_qt': rec_qt,
            'dir': sdir, 'mutation': cx['mutation'],
            'lig_coords': lig_sel.positions if len(lig_sel) > 0 else None,
        }
        print('OK')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\n=== {len(structures)}/{len(EGFR_STRUCTURES)} prepared ===')


In [ ]:
#@title 2-2. 구조 정렬 (WT 기준) {display-mode: "form"}
labels = list(structures.keys())
ref_label = labels[0]
ref_u = mda.Universe(structures[ref_label]['prot_H'])

for label in labels[1:]:
    try:
        mobile_u = mda.Universe(structures[label]['prot_H'])
        _, rmsd = mda_align.alignto(mobile_u, ref_u, select='name CA')
        aligned_path = structures[label]['prot_H'].replace('.pdb', '_aligned.pdb')
        mobile_u.atoms.write(aligned_path)
        structures[label]['prot_H_aligned'] = aligned_path
        print(f'{label} aligned to {ref_label}: RMSD={rmsd:.2f} Å')
    except Exception as e:
        print(f'{label} align failed: {e}')

structures[ref_label]['prot_H_aligned'] = structures[ref_label]['prot_H']


## 3. Docking Box


In [ ]:
#@title 3-1. Box 설정 {display-mode: "form"}
BOX_METHOD = "auto"  #@param ["auto", "residue", "manual"]
RESIDUE_LIST = "790, 858, 855"  #@param {type:"string"}
PADDING = 7.0  #@param {type:"number"}

def get_box_from_coords(coords, padding=7.0):
    minC, maxC = coords.min(axis=0), coords.max(axis=0)
    return (
        {'x': float((maxC[0]+minC[0])/2), 'y': float((maxC[1]+minC[1])/2), 'z': float((maxC[2]+minC[2])/2)},
        {'x': float(maxC[0]-minC[0]+2*padding), 'y': float(maxC[1]-minC[1]+2*padding), 'z': float(maxC[2]-minC[2]+2*padding)}
    )

for label, s in structures.items():
    if BOX_METHOD == "auto" and s['lig_coords'] is not None:
        s['center'], s['size'] = get_box_from_coords(s['lig_coords'], PADDING)
    elif BOX_METHOD == "residue":
        res_nums = [int(r.strip()) for r in RESIDUE_LIST.split(',')]
        ref = mda.Universe(s['prot_H'])
        sel = ref.select_atoms(' or '.join([f'resid {r}' for r in res_nums]))
        s['center'], s['size'] = get_box_from_coords(sel.positions if len(sel) > 0 else s['lig_coords'], PADDING)
    print(f'{label}: center=({s["center"]["x"]:.1f}, {s["center"]["y"]:.1f}, {s["center"]["z"]:.1f})')


In [ ]:
#@title 3-2. Box 3D 시각화 {display-mode: "form"}
first = list(structures.keys())[0]
s = structures[first]

view = py3Dmol.view(width=800, height=600)
with open(s['prot_H']) as f: view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.6}})
view.addBox({
    'center': {'x': s['center']['x'], 'y': s['center']['y'], 'z': s['center']['z']},
    'dimensions': {'w': s['size']['x'], 'h': s['size']['y'], 'd': s['size']['z']},
    'color': 'blue', 'opacity': 0.15
})
print(f'{first}: Box 시각화')
view.zoomTo()
view.show()


## 4. Cross-docking (약물 × 변이체)


In [ ]:
#@title 4-1. 전체 약물 × 변이체 도킹 {display-mode: "form"}
EXHAUSTIVENESS = 32  #@param {type:"integer"}
N_POSES = 5          #@param {type:"integer"}
N_CPUS = 10          #@param {type:"integer"}

smina = os.path.join(BIN_DIR, 'smina')
xdock_results = []
t0 = time.time()

# Prepare drug PDBQTs
drug_pdbqts = {}
for drug_name, drug_info in DRUGS.items():
    rdmol = Chem.MolFromSmiles(drug_info['smiles'])
    if rdmol is None: continue
    rdmol = Chem.AddHs(rdmol)
    AllChem.EmbedMolecule(rdmol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(rdmol)
    if len(Chem.GetMolFrags(rdmol)) != 1: continue
    
    lig_qt = os.path.join(WORK_DIR, f'{drug_name}.pdbqt')
    try:
        for setup in MoleculePreparation().prepare(rdmol):
            pdbqt_str, is_ok, _ = PDBQTWriterLegacy.write_string(setup)
            if is_ok:
                with open(lig_qt, 'w') as f: f.write(pdbqt_str)
                drug_pdbqts[drug_name] = lig_qt
                break
    except: pass

print(f'{len(drug_pdbqts)} drugs prepared')

# Cross-dock
total = len(drug_pdbqts) * len(structures)
done = 0

for drug_name, lig_qt in drug_pdbqts.items():
    for label, s in structures.items():
        done += 1
        print(f'[{done}/{total}] {drug_name} → {label}...', end=' ', flush=True)
        
        sdf_out = os.path.join(s['dir'], f'{drug_name}_docked.sdf')
        subprocess.run([
            smina, '-r', s['rec_qt'], '-l', lig_qt, '-o', sdf_out,
            '--center_x', str(s['center']['x']), '--center_y', str(s['center']['y']), '--center_z', str(s['center']['z']),
            '--size_x', str(s['size']['x']), '--size_y', str(s['size']['y']), '--size_z', str(s['size']['z']),
            '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(N_POSES),
            '--cpu', str(N_CPUS),
        ], capture_output=True)
        
        score = None
        if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
            suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
            if suppl and suppl[0]:
                props = suppl[0].GetPropsAsDict()
                if 'minimizedAffinity' in props:
                    try: score = float(props['minimizedAffinity'])
                    except: pass
        
        xdock_results.append({
            'Drug': drug_name, 'Structure': label, 'Mutation': s['mutation'],
            'Generation': DRUGS[drug_name]['gen'],
            'Score': round(score, 2) if score else None,
        })
        print(f'{score:.2f}' if score else 'FAILED')

elapsed = time.time() - t0
print(f'\n=== {len([r for r in xdock_results if r["Score"]])} / {total} docked in {elapsed:.0f}s ===')


## 5. 내성 패턴 분석


In [ ]:
#@title 5-1. Cross-docking 히트맵 {display-mode: "form"}
xdf = pd.DataFrame(xdock_results)

# Pivot: rows=structure, columns=drug
matrix = xdf.pivot_table(index='Structure', columns='Drug', values='Score', aggfunc='first')

# Annotate mutation on index
mut_map = {cx['label']: cx['mutation'] for cx in EGFR_STRUCTURES}
row_labels = [f"{l} ({mut_map.get(l, '')})" for l in matrix.index]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(matrix.astype(float), annot=True, fmt='.1f', cmap='RdYlGn',
            center=-7.0, ax=ax, linewidths=0.8, yticklabels=row_labels,
            cbar_kws={'label': 'Docking Score (kcal/mol)'})
ax.set_xlabel('Drug (→ newer generation)')
ax.set_ylabel('EGFR Structure (mutation)')
ax.set_title('EGFR Cross-docking: Drug Generation × Mutation')
plt.tight_layout()
plt.show()


In [ ]:
#@title 5-2. 약물 세대별 변이 감수성 {display-mode: "form"}
fig, ax = plt.subplots(figsize=(10, 5))

mut_colors = {'Wild-type': '#3498DB', 'L858R': '#E67E22', 'T790M': '#E74C3C', 'T790M/L858R': '#8E44AD'}

for mut in xdf['Mutation'].unique():
    sub = xdf[xdf['Mutation'] == mut].dropna(subset=['Score'])
    if sub.empty: continue
    avg = sub.groupby('Drug')['Score'].mean()
    ax.plot(avg.index, avg.values, 'o-', label=mut, color=mut_colors.get(mut, 'gray'), markersize=8, linewidth=2)

ax.set_ylabel('Docking Score (kcal/mol)')
ax.set_title('Drug Sensitivity by EGFR Mutation')
ax.legend()
ax.axhline(y=-7.0, color='gray', linestyle='--', alpha=0.3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\n관찰 포인트:')
print('• 1세대(Erlotinib/Gefitinib): WT/L858R에 효과적, T790M에 약함')
print('• 3세대(Osimertinib): T790M에 선택적')


## 6. 내보내기


In [ ]:
#@title 6-1. 결과 내보내기 {display-mode: "form"}
import shutil

csv_path = os.path.join(WORK_DIR, 'egfr_crossdock.csv')
xdf.to_csv(csv_path, index=False)
matrix.to_csv(os.path.join(WORK_DIR, 'egfr_matrix.csv'))
print(f'CSV: {csv_path}')

pml_path = os.path.join(WORK_DIR, 'egfr_pymol.pml')
with open(pml_path, 'w') as f:
    f.write('reinitialize\nbg_color white\n\n')
    pml_colors = ['palegreen','salmon','lightblue','lightorange']
    for i, (label, s) in enumerate(structures.items()):
        prot = s.get('prot_H_aligned', s['prot_H'])
        f.write(f'load {prot}, {label}\ncolor {pml_colors[i%len(pml_colors)]}, {label}\nshow cartoon, {label}\n\n')
print(f'PyMOL: {pml_path}')

zip_path = os.path.join(os.path.dirname(WORK_DIR), 'egfr_mutants_results')
shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'Archive: {zip_path}.zip')
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Results at: {WORK_DIR}')
